## Imports and Supabase Connection

In [70]:
import os
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv
from pathlib import Path
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf

%matplotlib inline
# Resolve the project root (one level above `/notebooks`).
project_root = Path.cwd().parent
env_path = project_root / ".env"

load_dotenv(dotenv_path=env_path, override=True)

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Optional sanity check.
print("Loaded key starts with:", SUPABASE_KEY[:5])


Loaded key starts with: eyJhb


## Incident

In [71]:
# Read the incident table in 1,000-row pages to avoid timeouts and memory spikes.
page_size = 1000
# Accumulate rows until Supabase returns an empty page.
all_rows = []
start = 0

while True:
    response = (
        supabase
        .table("incident")
        .select("*")
        .order("Incident_ID")   # Required for stable pagination.
        .range(start, start + page_size - 1)
        .execute()
    )

    data = response.data
    if not data:
        break

    all_rows.extend(data)
    start += page_size

incident_df = pd.DataFrame(all_rows)
incident_df["Number_Victims"] = pd.to_numeric(incident_df["Number_Victims"], errors="coerce")

# Quick validation of total row count.
print (len(incident_df))
print (incident_df.head(2))


3136
     Incident_ID  Month  Day  Year                       Date  \
0  19660311NCIRC      3   11  1966  1966-03-11T00:00:00+00:00   
1  19660314TXCAW      3   14  1966  1966-03-14T00:00:00+00:00   

                             School Victims_Killed Victims_Wounded  \
0  Irwing Avenue Junior High School              0               1   
1                Carver High School              1               0   

   Number_Victims Shooter_Killed  ... Preplanned SRO_School  \
0               1              0  ...         No              
1               1              0  ...         No        Yes   

  Security_Screening     Screening_Outcome Shots_Fired School_Lockdown  \
0                                                    7                   
1       Armed Guards  Outside/Off-Property           3                   

         LAT         LNG Campus_Type Zipcode  
0  35.237069  -80.850227               28202  
1   31.57954  -97.130303               76704  

[2 rows x 50 columns]


# Model Exploring

## Incidents

### Apply the best subset selection approach to incident data. We wish to predict risk based on a variety of features/predictors

In [72]:
valid_states = {
"AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA",
"HI","ID","IL","IN","IA","KS","KY","LA","ME","MD",
"MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ",
"NM","NY","NC","ND","OH","OK","OR","PA","RI","SC",
"SD","TN","TX","UT","VT","VA","WA","WV","WI","WY","DC"
}

invalid = incident_df[~incident_df["State"].isin(valid_states)]
invalid["State"].unique()

<StringArray>
[             'Winter',             'Houston',         'Rogersville',
           'Hahnville',             'Artesia',           'Pensacola',
           'St. Louis',              'Dallas',          'Fort Worth',
              'Spring',                'Fall',            'Hartford',
            'Anniston',               'Miami',              'Aldine',
            'Portland',        'New Rochelle',        'Indianapolis',
               'Tampa',             'Bristol',          'Fort Myers',
            'Winnetka',             'Detroit',                'Oahu',
         'Baton Rouge',         'Port Arthur',      'Virginia Beach',
            'Stockton',          'Washington',             'Slidell',
           'Rock Hill',              'Merced',            'Amarillo',
          'Montgomery',                   '4',              'Summer',
 'San Fernando Valley',          'Manchester',            'New York',
               'Omaha',                  'VI',                    '',
      

## Sample Restriction (1987–Present)

The panel is restricted to 1987 onward due to a structural break in the underlying data-generating process prior to 1987.

Pre-1987 observations are not comparable in measurement, reporting coverage, and policy codification. Including those years would violate panel homogeneity assumptions and introduce non-structural noise into fixed effects and dynamic specifications.

This restriction is not outcome-based filtering. It is a data consistency adjustment to ensure:

- Comparable incident classification
- Consistent enrollment coverage
- Reliable policy coding
- Stable variance structure

All models are therefore estimated on the 1987–present balanced policy-enrollment panel.


## Incidents from 1987

In [73]:
incident_df_from_1987 = incident_df[incident_df["State"].isin(valid_states)]
print(incident_df_from_1987.shape) # incident_df_from_1987.shape
print(incident_df_from_1987["State"].unique())

(3070, 50)
<StringArray>
['NC', 'TX', 'CA', 'NY', 'FL', 'PA', 'MN', 'TN', 'IL', 'VT', 'HI', 'OK', 'KY',
 'DC', 'WI', 'OH', 'AR', 'DE', 'UT', 'ID', 'IA', 'MI', 'NJ', 'AZ', 'VA', 'MD',
 'NM', 'LA', 'NV', 'IN', 'AL', 'MO', 'CT', 'SC', 'GA', 'CO', 'MA', 'WV', 'MS',
 'OR', 'MT', 'KS', 'WA', 'WY', 'RI', 'NH', 'NE', 'AK', 'ME', 'SD', 'ND']
Length: 51, dtype: str


In [74]:
# incident count per state-year combination
incident_counts = (
    incident_df_from_1987
    .groupby(["State", "Year"])
    .size()
    .reset_index(name="incident_count")
)

# Ensure Victims_Killed is numeric before aggregation
incident_df_from_1987["Victims_Killed"] = pd.to_numeric(
    incident_df_from_1987["Victims_Killed"],
    errors="coerce"
)

# Replace any non-numeric values with 0
incident_df_from_1987["Victims_Killed"] = (
    incident_df_from_1987["Victims_Killed"].fillna(0)
)

In [75]:
# supabase.table("state_year_enrollment").select("*").execute()

In [76]:
response = supabase.table("Enrollment Data Final").select("*").limit(1).execute(); print(response.data)

[{'school_id': 601195, 'state': 'California', 'year': 1996, 'total_students': 154}]


In [77]:
all_rows = []
start = 0
batch_size = 1000

while True:
    response = (
        supabase
        .table("enrollment_state_year_mat")
        .select("*")
        .range(start, start + batch_size - 1)
        .execute()
    )
    
    data = response.data
    
    if not data:
        break
        
    all_rows.extend(data)
    start += batch_size

enrollment_df = pd.DataFrame(all_rows)

print(enrollment_df.shape)

(2548, 3)


In [78]:
# Keep only incident years that exist in enrollment data
incident_counts = incident_counts[
    incident_counts["Year"] >= 1987
]

In [79]:
enrollment_df


,state,year,total_students
0,RHODE ISLAND,2020,10006
1,Louisiana,2025,569440
2,Montana,1999,160294
3,Georgia,2015,1721666
4,Rhode Island,2016,139409
...,...,...,...
2543,District of Columbia,2013,75819
2544,Kentucky,2014,683957
2545,GEORGIA,2020,87127
2546,DISTRICT OF COLUMBIA,2015,1671


In [80]:
# Convert full state names to uppercase for consistent mapping
enrollment_df["state"] = enrollment_df["state"].str.upper()


# Map full state names to two-letter abbreviations
state_abbrev = {
    'ALABAMA': 'AL', 'ALASKA': 'AK', 'ARIZONA': 'AZ', 'ARKANSAS': 'AR',
    'CALIFORNIA': 'CA', 'COLORADO': 'CO', 'CONNECTICUT': 'CT',
    'DELAWARE': 'DE', 'DISTRICT OF COLUMBIA': 'DC', 'FLORIDA': 'FL',
    'GEORGIA': 'GA', 'HAWAII': 'HI', 'IDAHO': 'ID', 'ILLINOIS': 'IL',
    'INDIANA': 'IN', 'IOWA': 'IA', 'KANSAS': 'KS', 'KENTUCKY': 'KY',
    'LOUISIANA': 'LA', 'MAINE': 'ME', 'MARYLAND': 'MD',
    'MASSACHUSETTS': 'MA', 'MICHIGAN': 'MI', 'MINNESOTA': 'MN',
    'MISSISSIPPI': 'MS', 'MISSOURI': 'MO', 'MONTANA': 'MT',
    'NEBRASKA': 'NE', 'NEVADA': 'NV', 'NEW HAMPSHIRE': 'NH',
    'NEW JERSEY': 'NJ', 'NEW MEXICO': 'NM', 'NEW YORK': 'NY',
    'NORTH CAROLINA': 'NC', 'NORTH DAKOTA': 'ND', 'OHIO': 'OH',
    'OKLAHOMA': 'OK', 'OREGON': 'OR', 'PENNSYLVANIA': 'PA',
    'RHODE ISLAND': 'RI', 'SOUTH CAROLINA': 'SC',
    'SOUTH DAKOTA': 'SD', 'TENNESSEE': 'TN', 'TEXAS': 'TX',
    'UTAH': 'UT', 'VERMONT': 'VT', 'VIRGINIA': 'VA',
    'WASHINGTON': 'WA', 'WEST VIRGINIA': 'WV',
    'WISCONSIN': 'WI', 'WYOMING': 'WY'
}

# Create State column matching incident dataset
enrollment_df["State"] = enrollment_df["state"].map(state_abbrev)

# Rename year column for merge consistency
enrollment_df = enrollment_df.rename(columns={"year": "Year"})

In [81]:
# Keep only the larger enrollment value per state–year
enrollment_df = (
    enrollment_df
    .sort_values("total_students", ascending=False)
    .drop_duplicates(["State", "Year"])
)

In [82]:
# Merge incident counts with enrollment data on State and Year
merged_df_table = incident_counts.merge(
    enrollment_df[["State", "Year", "total_students"]],
    on=["State", "Year"],
    how="left"
)

# Check for any unmatched rows after merge
print(merged_df_table["total_students"].isna().sum())

# Compute incident rate per 100,000 students
merged_df_table["incident_rate_per_100k"] = (
    merged_df_table["incident_count"] / merged_df_table["total_students"]
) * 100000

# Inspect result
print(merged_df_table.head())

0
  State  Year  incident_count  total_students  incident_rate_per_100k
0    AK  1997               1          129919                0.769710
1    AK  2005               1          131701                0.759296
2    AK  2018               1          129660                0.771248
3    AK  2019               1          127641                0.783447
4    AK  2022               1          126457                0.790783


This notebook focuses on state-year incident risk, not conditional harm severity.


In [83]:
incident_df_from_1987 = incident_df_from_1987[incident_df_from_1987["State"].str.match(r"^[A-Z]{2}$", na=False)]
print("Remaining rows:", len(incident_df_from_1987))

Remaining rows: 3070


In [84]:
valid_states = incident_df_from_1987["State"].str.match(r"^[A-Z]{2}$", na=False)
print("Valid rows:", valid_states.sum())
print("Invalid rows:", (~valid_states).sum())

Valid rows: 3070
Invalid rows: 0


In [85]:
print(incident_df_from_1987.head())

     Incident_ID  Month  Day  Year                       Date  \
0  19660311NCIRC      3   11  1966  1966-03-11T00:00:00+00:00   
1  19660314TXCAW      3   14  1966  1966-03-14T00:00:00+00:00   
2  19660324CACAM      3   24  1966  1966-03-24T00:00:00+00:00   
3  19660328CAJOL      3   28  1966  1966-03-28T00:00:00+00:00   
4  19660427NYBAB      4   27  1966  1966-04-27T00:00:00+00:00   

                             School  Victims_Killed Victims_Wounded  \
0  Irwing Avenue Junior High School               0               1   
1                Carver High School               1               0   
2    Camino Pablo Elementary School               3               0   
3                Jordan High School               0               1   
4             Bay Shore High School               1               0   

   Number_Victims Shooter_Killed  ... Preplanned SRO_School  \
0               1              0  ...         No              
1               1              0  ...         No        

In [86]:
print(merged_df_table["incident_count"].describe())

count    861.000000
mean       3.191638
std        3.745599
min        1.000000
25%        1.000000
50%        2.000000
75%        4.000000
max       25.000000
Name: incident_count, dtype: float64


In [87]:
print(merged_df_table["total_students"].describe())

count    8.610000e+02
mean     1.339581e+06
std      1.313322e+06
min      5.003200e+04
25%      5.573810e+05
50%      8.762130e+05
75%      1.662643e+06
max      6.322586e+06
Name: total_students, dtype: float64


In [88]:
import numpy as np

merged_df_table["log_enrollment"] = np.log(merged_df_table["total_students"])
print(merged_df_table["log_enrollment"].describe())

count    861.000000
mean      13.701213
std        0.947233
min       10.820418
25%       13.231004
50%       13.683364
75%       14.323919
max       15.659639
Name: log_enrollment, dtype: float64


In [89]:
incident_agg = (
    incident_df_from_1987
    .groupby(["State","Year"], as_index=False)
    .size()
    .rename(columns={"size": "incident_count"})
)

merged_df_table = enrollment_df.merge(
    incident_agg,
    on=["State","Year"],
    how="left"
)

merged_df_table["incident_count"] = merged_df_table["incident_count"].fillna(0)

print(merged_df_table.shape)
print((merged_df_table["incident_count"] == 0).sum())

(1989, 5)
1128


In [90]:
merged_df_table["log_enrollment"] = np.log(merged_df_table["total_students"])
print(merged_df_table.head())

        state  Year  total_students State  incident_count  log_enrollment
0  CALIFORNIA  2005         6322586    CA             6.0       15.659639
1  CALIFORNIA  2006         6312103    CA             4.0       15.657979
2  CALIFORNIA  2004         6298928    CA             7.0       15.655890
3  CALIFORNIA  2007         6274813    CA             5.0       15.652054
4  CALIFORNIA  2003         6244403    CA             4.0       15.647196


In [91]:
merged_df_table = merged_df_table.drop(columns=["state"])
print(merged_df_table.columns)

Index(['Year', 'total_students', 'State', 'incident_count', 'log_enrollment'], dtype='str')


In [92]:
all_rows = []
start = 0
batch_size = 1000

while True:
    response = (
        supabase
        .table("background_check_binary")
        .select("*")
        .range(start, start + batch_size - 1)
        .execute()
    )
    
    data = response.data
    
    if not data:
        break
        
    all_rows.extend(data)
    start += batch_size

bg_df = pd.DataFrame(all_rows)

print(bg_df.shape)

(2295, 3)


In [93]:
policy_tables = [
    "background_check_binary",
    "permit_to_purchase_binary",
    "prohibited_possessor_binary",
    "waiting_period_binary",
    "registration_binary",
    "child_access_binary",
    "safety_training_binary",
    "report_stolen_lost_binary",
    "firearm_sales_retrictions_binary",
    "k12_settings_binary"
]

policy_dfs = {}

for table in policy_tables:
    all_rows = []
    start = 0
    batch_size = 1000

    while True:
        response = (
            supabase
            .table(table)
            .select("*")
            .range(start, start + batch_size - 1)
            .execute()
        )
        data = response.data
        if not data:
            break
        all_rows.extend(data)
        start += batch_size

    policy_dfs[table] = pd.DataFrame(all_rows)

print({k: v.shape for k, v in policy_dfs.items()})

{'background_check_binary': (2295, 3), 'permit_to_purchase_binary': (2115, 3), 'prohibited_possessor_binary': (2115, 3), 'waiting_period_binary': (2295, 3), 'registration_binary': (2295, 3), 'child_access_binary': (1620, 3), 'safety_training_binary': (405, 3), 'report_stolen_lost_binary': (765, 3), 'firearm_sales_retrictions_binary': (2295, 3), 'k12_settings_binary': (315, 3)}


In [94]:
for name, df in policy_dfs.items():
    merged_df_table = merged_df_table.merge(
        df,
        left_on=["State","Year"],
        right_on=["state","year"],
        how="left"
    )
    merged_df_table = merged_df_table.drop(columns=["state","year"])

# Replace NaN with 0 for all policy indicators
policy_cols = [col for col in merged_df_table.columns if col.endswith("_law")]
merged_df_table[policy_cols] = merged_df_table[policy_cols].fillna(0)

print(merged_df_table.shape)
print(len(policy_cols), "policy variables merged")

(1989, 15)
10 policy variables merged


## State-Year Risk Model

### Objective

Estimate the association between state firearm policies and the **risk of school shooting incidents**, measured at the state-year level.

Risk is operationalized as the incident rate:

\[
\text{Risk}_{s,t} =
\frac{\text{Incident Count}_{s,t}}{\text{Student Enrollment}_{s,t}}
\]

For descriptive visuals, this rate is scaled to incidents per 100,000 students.

---

### Data Structure

The analysis uses a state-year panel with:

- All U.S. states plus DC  
- Years in the study window  
- Zero-incident state-years retained  
- Total student enrollment as the exposure measure  
- Binary firearm policy indicators  

Each observation is one **state-year**.

---

### Modeling Strategy

A **Negative Binomial regression** is used because the dependent variable is a nonnegative count with overdispersion and many zeros.

The model is estimated with:

- Outcome: `incident_count`  
- Offset: `log_enrollment`  
- Policy indicators as covariates  

Including `log(total_students)` as an offset means the model is estimating the **incident rate**, not a raw count relationship.

---

### Specification

\[
\log(E[Y_{s,t}]) =
\beta_0 +
\beta_k \text{Policy}_{k,s,t} +
\log(\text{Enrollment}_{s,t})
\]

where `Y_{s,t}` is the number of incidents in state `s` and year `t`. Rearranging the offset form shows that the model is equivalent to modeling risk per student.

---

### Interpretation

Exponentiated coefficients are **Incidence Rate Ratios (IRRs)**:

- IRR < 1 implies lower incident risk  
- IRR > 1 implies higher incident risk  

Substantively, the question is whether policies are associated with fewer incidents relative to enrollment exposure.

---

### Scope

This notebook is limited to **risk of incident occurrence**. Outcomes based on victim counts or victims per incident are intentionally excluded from the main framework.


In [95]:
formula = """
incident_count ~ 
background_check_law +
permit_to_purchase_law +
prohibited_possessor_law +
waiting_period_law +
registration_law +
child_access_law +
safety_training_law +
report_stolen_lost_law +
firearm_sales_retrictions_law +
k12_settings_law +
C(State) +
C(Year)
"""

model_joint = smf.glm(
    formula=formula,
    data=merged_df_table,
    family=sm.families.NegativeBinomial(),
    offset=merged_df_table["log_enrollment"]
).fit()

print(model_joint.summary())

/home/tommie-clark/capstoneProje t/capstoneProject2026/.venv/lib/python3.12/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


                 Generalized Linear Model Regression Results                  
Dep. Variable:         incident_count   No. Observations:                 1989
Model:                            GLM   Df Residuals:                     1890
Model Family:        NegativeBinomial   Df Model:                           98
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2247.4
Date:                Wed, 08 Apr 2026   Deviance:                       1055.1
Time:                        12:47:49   Pearson chi2:                 1.36e+03
No. Iterations:                     9   Pseudo R-squ. (CS):             0.4500
Covariance Type:            nonrobust                                         
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept     

In [96]:
import numpy as np
print(np.exp(model_joint.params))

Intercept                        2.811915e-07
C(State)[T.AL]                   2.009817e+00
C(State)[T.AR]                   1.349513e+00
C(State)[T.AZ]                   6.003576e-01
C(State)[T.CA]                   1.453006e+00
                                     ...     
child_access_law                 1.045549e+00
safety_training_law              8.908755e-01
report_stolen_lost_law           7.385070e-01
firearm_sales_retrictions_law    1.009689e+00
k12_settings_law                 6.729423e-01
Length: 99, dtype: float64


## Two-Way Fixed Effects Risk Model

### Objective

Strengthen the risk model by controlling for both:

- **State fixed effects** for time-invariant state characteristics
- **Year fixed effects** for nationwide shocks and trend shifts

This produces a two-way fixed effects specification for **enrollment-adjusted incident risk**.

---

### Why Add Year Fixed Effects?

Year effects absorb national movements in school shooting risk that are shared across states, including:

- Broad upward time trends  
- Reporting and classification changes  
- National policy and media shocks  
- Other macro-level temporal shifts  

Without year controls, policy coefficients can incorrectly absorb nationwide trend effects.

---

### Specification

The model includes:

- `C(State)` for baseline state differences  
- `C(Year)` for common year shocks  
- Binary policy indicators  
- `log(total_students)` as the exposure offset  

\[
\log(E[Y_{s,t}]) =
\beta_0 +
\beta_k \text{Policy}_{k,s,t} +
\gamma_s +
\delta_t +
\log(\text{Enrollment}_{s,t})
\]

---

### Interpretation

Policy coefficients now represent **within-state changes in incident risk**, after accounting for nationwide year shocks and state-specific baselines.

This is the core risk specification for the project.

---

### Methodological Implication

The TWFE risk model improves identification by separating policy timing from broad national trend movement. Results should still be interpreted as associative rather than causal because policy adoption may respond to prior incidents.


In [97]:
formula = """
incident_count ~ 
background_check_law +
permit_to_purchase_law +
prohibited_possessor_law +
waiting_period_law +
registration_law +
child_access_law +
safety_training_law +
report_stolen_lost_law +
firearm_sales_retrictions_law +
k12_settings_law +
C(State) +
C(Year)
"""

model_twfe = smf.glm(
    formula=formula,
    data=merged_df_table,
    family=sm.families.NegativeBinomial(),
    offset=merged_df_table["log_enrollment"]
).fit()

print(model_twfe.summary())


/home/tommie-clark/capstoneProje t/capstoneProject2026/.venv/lib/python3.12/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


                 Generalized Linear Model Regression Results                  
Dep. Variable:         incident_count   No. Observations:                 1989
Model:                            GLM   Df Residuals:                     1890
Model Family:        NegativeBinomial   Df Model:                           98
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2247.4
Date:                Wed, 08 Apr 2026   Deviance:                       1055.1
Time:                        12:47:52   Pearson chi2:                 1.36e+03
No. Iterations:                     9   Pseudo R-squ. (CS):             0.4500
Covariance Type:            nonrobust                                         
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept     

## Cluster-Robust Inference (State-Level)


### Why Cluster Standard Errors?

This is a state × year panel, so errors can be serially correlated within states across years.
Clustering by state accounts for within-state dependence in the residual process.
This changes inference (standard errors and p-values), not coefficient point estimates.
State-clustered standard errors are typically more conservative and methodologically appropriate for this panel structure.


In [98]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

# ================================
# STEP 1 — Fit Poisson TWFE Model
# ================================

model_twfe = smf.glm(
    formula=formula,
    data=merged_df_table,
    family=sm.families.Poisson(),
    offset=merged_df_table["log_enrollment"]
).fit()

# ================================
# STEP 2 — Fit Clustered Version
# ================================

model_twfe_cluster = smf.glm(
    formula=formula,
    data=merged_df_table,
    family=sm.families.Poisson(),
    offset=merged_df_table["log_enrollment"]
).fit(
    cov_type="cluster",
    cov_kwds={"groups": merged_df_table["State"]}
)

print("Two-Way FE Poisson with State-Clustered SE")
print(model_twfe_cluster.summary())

# ================================
# STEP 3 — Policy-Only Comparison
# ================================

policy_variables = [v for v in model_twfe.params.index if v.endswith("_law")]

se_comparison = pd.DataFrame({
    "variable": policy_variables,
    "coef": [model_twfe.params[v] for v in policy_variables],
    "se_nonclustered": [model_twfe.bse[v] for v in policy_variables],
    "se_clustered": [model_twfe_cluster.bse[v] for v in policy_variables],
    "p_nonclustered": [model_twfe.pvalues[v] for v in policy_variables],
    "p_clustered": [model_twfe_cluster.pvalues[v] for v in policy_variables],
})

print("\nStandard Error Comparison")
print(se_comparison)



Two-Way FE Poisson with State-Clustered SE
                 Generalized Linear Model Regression Results                  
Dep. Variable:         incident_count   No. Observations:                 1989
Model:                            GLM   Df Residuals:                     1890
Model Family:                 Poisson   Df Model:                           98
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2072.7
Date:                Wed, 08 Apr 2026   Deviance:                       1831.5
Time:                        12:47:57   Pearson chi2:                 2.22e+03
No. Iterations:                     8   Pseudo R-squ. (CS):             0.8208
Covariance Type:              cluster                                         
                                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------

## Lag the policy variables.

We test whether policy adoption predicts future incident risk rather than same-year effects.

In [99]:
policy_cols = [
    "background_check_law",
    "permit_to_purchase_law",
    "prohibited_possessor_law",
    "waiting_period_law",
    "registration_law",
    "child_access_law",
    "safety_training_law",
    "report_stolen_lost_law",
    "firearm_sales_retrictions_law",
    "k12_settings_law"
]

# sort properly first
merged_df_table = merged_df_table.sort_values(["State", "Year"])

# create 1-year lags within state
for col in policy_cols:
    merged_df_table[col + "_lag1"] = (
        merged_df_table.groupby("State")[col].shift(1)
    )

print(
    merged_df_table[[c for c in merged_df_table.columns if "_lag1" in c]].isna().sum().head()
)

background_check_law_lag1        51
permit_to_purchase_law_lag1      51
prohibited_possessor_law_lag1    51
waiting_period_law_lag1          51
registration_law_lag1            51
dtype: int64


In [100]:
lag_cols = [col for col in merged_df_table.columns if col.endswith("_lag1")]

merged_df_table[lag_cols] = merged_df_table[lag_cols].fillna(0)

print(merged_df_table[lag_cols].isna().sum().sum())

0


Notice the Year coefficients:

2018–2024 effects are extremely large and significant.

This indicates:

National upward structural shift in incidents dominates policy variation.

That is your strongest empirical finding so far.

evidence currently supports:

Strong national time dynamics

Weak within-state short-run policy effects

Significant endogeneity concerns

That is a legitimate, publishable conclusion if framed correctly.

In [101]:
formula_lag = """
incident_count ~ 
background_check_law_lag1 +
permit_to_purchase_law_lag1 +
prohibited_possessor_law_lag1 +
waiting_period_law_lag1 +
registration_law_lag1 +
child_access_law_lag1 +
safety_training_law_lag1 +
report_stolen_lost_law_lag1 +
firearm_sales_retrictions_law_lag1 +
k12_settings_law_lag1 +
C(State) +
C(Year)
"""

model_twfe_lag = smf.glm(
    formula=formula_lag,
    data=merged_df_table,
    family=sm.families.NegativeBinomial(),
    offset=merged_df_table["log_enrollment"]
).fit()

print(model_twfe_lag.summary())

/home/tommie-clark/capstoneProje t/capstoneProject2026/.venv/lib/python3.12/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


                 Generalized Linear Model Regression Results                  
Dep. Variable:         incident_count   No. Observations:                 1989
Model:                            GLM   Df Residuals:                     1890
Model Family:        NegativeBinomial   Df Model:                           98
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2251.4
Date:                Wed, 08 Apr 2026   Deviance:                       1063.2
Time:                        12:47:58   Pearson chi2:                 1.34e+03
No. Iterations:                     9   Pseudo R-squ. (CS):             0.4478
Covariance Type:            nonrobust                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Inte

## Risk-Only Scope

The notebook remains limited to enrollment-adjusted incident risk models.

- Primary outcome: `incident_count`  
- Exposure adjustment: `log(total_students)` offset  
- Substantive target: lower risk of incident occurrence  

Severity outcomes such as `total_victims` and `victims_per_incident` are excluded from the main modeling framework.


## DC Exclusion Robustness Check

Refit the negative binomial risk models after excluding DC and compare policy coefficients across the full sample and the no-DC sample.

The two non-lag negative binomial cells above use the same specification, so the DC robustness refit is reported once for that specification.

The final comparison table below includes only:

- `nb_twfe`
- `nb_twfe_lagged`


In [102]:
from IPython.display import display

merged_df_table_no_dc = merged_df_table[merged_df_table["State"] != "DC"].copy()

sample_comparison = pd.DataFrame(
    {
        "sample": ["full", "no_dc"],
        "observations": [len(merged_df_table), len(merged_df_table_no_dc)],
        "states": [merged_df_table["State"].nunique(), merged_df_table_no_dc["State"].nunique()],
    }
)

def fit_risk_glm(formula_text, data, family, cluster=False):
    glm = smf.glm(
        formula=formula_text,
        data=data,
        family=family,
        offset=data["log_enrollment"],
    )
    if cluster:
        return glm.fit(cov_type="cluster", cov_kwds={"groups": data["State"]})
    return glm.fit()

def compare_policy_coefficients(full_model, no_dc_model, terms, model_name):
    comparison = pd.DataFrame(
        {
            "model": model_name,
            "term": terms,
            "coef_full": full_model.params.reindex(terms).values,
            "coef_no_dc": no_dc_model.params.reindex(terms).values,
            "irr_full": np.exp(full_model.params.reindex(terms).values),
            "irr_no_dc": np.exp(no_dc_model.params.reindex(terms).values),
            "p_full": full_model.pvalues.reindex(terms).values,
            "p_no_dc": no_dc_model.pvalues.reindex(terms).values,
        }
    )
    comparison["delta_coef"] = comparison["coef_no_dc"] - comparison["coef_full"]
    comparison["sig_full_5pct"] = comparison["p_full"] < 0.05
    comparison["sig_no_dc_5pct"] = comparison["p_no_dc"] < 0.05
    return comparison

display(sample_comparison)


,sample,observations,states
0,full,1989,51
1,no_dc,1950,50


In [103]:
model_nb_full = fit_risk_glm(formula, merged_df_table, sm.families.NegativeBinomial())
model_nb_no_dc = fit_risk_glm(formula, merged_df_table_no_dc, sm.families.NegativeBinomial())

model_lag_full = fit_risk_glm(formula_lag, merged_df_table, sm.families.NegativeBinomial())
model_lag_no_dc = fit_risk_glm(formula_lag, merged_df_table_no_dc, sm.families.NegativeBinomial())


/home/tommie-clark/capstoneProje t/capstoneProject2026/.venv/lib/python3.12/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/home/tommie-clark/capstoneProje t/capstoneProject2026/.venv/lib/python3.12/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/home/tommie-clark/capstoneProje t/capstoneProject2026/.venv/lib/python3.12/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/home/tommie-clark/capstoneProje t/capstoneProject2026/.venv/lib/python3.12/site-packages/statsmo

In [104]:
base_policy_terms = [term for term in model_nb_full.params.index if term.endswith("_law")]
lag_policy_terms = [term for term in model_lag_full.params.index if term.endswith("_law_lag1")]

combined_policy_comparison = pd.concat(
    [
        compare_policy_coefficients(model_nb_full, model_nb_no_dc, base_policy_terms, "nb_twfe"),
        compare_policy_coefficients(model_lag_full, model_lag_no_dc, lag_policy_terms, "nb_twfe_lagged"),
    ],
    ignore_index=True,
)

combined_policy_comparison["abs_delta_coef"] = combined_policy_comparison["delta_coef"].abs()
combined_policy_comparison = combined_policy_comparison.sort_values(["model", "abs_delta_coef"], ascending=[True, False])

significance_change_summary = combined_policy_comparison[
    combined_policy_comparison["sig_full_5pct"] != combined_policy_comparison["sig_no_dc_5pct"]
].copy()

policy_signal_summary = combined_policy_comparison[
    combined_policy_comparison["sig_full_5pct"] | combined_policy_comparison["sig_no_dc_5pct"]
].copy()

display(combined_policy_comparison.round(4))
display(significance_change_summary.round(4))
display(policy_signal_summary.round(4))


,model,term,coef_full,coef_no_dc,irr_full,irr_no_dc,p_full,p_no_dc,delta_coef,sig_full_5pct,sig_no_dc_5pct,abs_delta_coef
6,nb_twfe,safety_training_law,-0.1156,0.0327,0.8909,1.0332,0.6085,0.8934,0.1482,False,False,0.1482
7,nb_twfe,report_stolen_lost_law,-0.3031,-0.3673,0.7385,0.6926,0.0839,0.0406,-0.0641,False,True,0.0641
8,nb_twfe,firearm_sales_retrictions_law,0.0096,0.0659,1.0097,1.0682,0.9549,0.7039,0.0563,False,False,0.0563
3,nb_twfe,waiting_period_law,-1.4190,-1.3837,0.2420,0.2506,0.1751,0.1862,0.0353,False,False,0.0353
4,nb_twfe,registration_law,1.6531,1.6201,5.2233,5.0537,0.1165,0.1241,-0.0330,False,False,0.0330
2,nb_twfe,prohibited_possessor_law,0.2047,0.1879,1.2272,1.2067,0.2004,0.2413,-0.0169,False,False,0.0169
9,nb_twfe,k12_settings_law,-0.3961,-0.3811,0.6729,0.6831,0.1268,0.1422,0.0150,False,False,0.0150
0,nb_twfe,background_check_law,-0.4003,-0.3863,0.6701,0.6796,0.0897,0.1055,0.0141,False,False,0.0141
5,nb_twfe,child_access_law,0.0445,0.0326,1.0455,1.0331,0.7787,0.8387,-0.0119,False,False,0.0119
1,nb_twfe,permit_to_purchase_law,0.2398,0.2515,1.2710,1.2860,0.1523,0.1374,0.0118,False,False,0.0118


,model,term,coef_full,coef_no_dc,irr_full,irr_no_dc,p_full,p_no_dc,delta_coef,sig_full_5pct,sig_no_dc_5pct,abs_delta_coef
7,nb_twfe,report_stolen_lost_law,-0.3031,-0.3673,0.7385,0.6926,0.0839,0.0406,-0.0641,False,True,0.0641


,model,term,coef_full,coef_no_dc,irr_full,irr_no_dc,p_full,p_no_dc,delta_coef,sig_full_5pct,sig_no_dc_5pct,abs_delta_coef
7,nb_twfe,report_stolen_lost_law,-0.3031,-0.3673,0.7385,0.6926,0.0839,0.0406,-0.0641,False,True,0.0641
